# 06 – Bulk Upload Local Units (Templates & Auto-Fill)

**Environment:** Staging (`goadmin-stage.ifrc.org`)

**Template row structure:**
- Row 1: Category group headings (e.g. 'Administrative Details')
- Row 2: Column names / field names (used as headers for data)
- Row 3: Additional info per column (e.g. mandatory, data type)
- Row 4+: Data rows to fill in

**Sheet 'Options':** Provides allowed values for dropdown columns.

In [55]:
%pip install -q openpyxl requests

import requests
import getpass
import os
import json
import openpyxl
from datetime import datetime

BASE_URL = "https://goadmin-stage.ifrc.org/api/v2"
TOKEN = getpass.getpass("Enter your IFRC GO STAGING API token: ")
HEADERS = {"Authorization": f"Token {TOKEN}", "Accept": "*/*"}

os.makedirs("output", exist_ok=True)
print("Setup complete.")

Note: you may need to restart the kernel to use updated packages.


c:\Users\arun.gandhi\Downloads\local_units_scripts\.venv\Scripts\python.exe: No module named pip


Setup complete.


## 1. Download Bulk Upload Templates
The API returns a JSON `{"template_url": "https://..."}`. We parse that and do a second request to get the actual `.xlsm` file.

In [56]:
templates_to_download = [
    {"type": "health_care", "filename": "output/template_health_care.xlsm"},
    {"type": "local_unit",  "filename": "output/template_local_unit.xlsm"}
]

for info in templates_to_download:
    print(f"\n--- {info['type']} ---")

    # Step 1: hit the API endpoint to get the JSON with the real URL
    api_url = f"{BASE_URL}/bulk-upload-local-unit/get-bulk-upload-template/?bulk_upload_template={info['type']}"
    r1 = requests.get(api_url, headers=HEADERS)
    print(f"Step 1 status: {r1.status_code}  body: {r1.text[:300]}")

    if r1.status_code != 200:
        print(f"FAILED at Step 1")
        continue

    # Step 2: parse JSON, extract template_url
    try:
        payload = r1.json()
    except ValueError:
        # Response was the raw file already
        with open(info['filename'], 'wb') as fh:
            fh.write(r1.content)
        print(f"Saved raw bytes -> {info['filename']}")
        continue

    template_url = payload.get("template_url")
    if not template_url:
        print(f"'template_url' missing: {payload}")
        continue

    print(f"Step 2 template_url: {template_url}")

    # Step 3: download the actual xlsm file
    r2 = requests.get(template_url)
    print(f"Step 3 status: {r2.status_code}  bytes received: {len(r2.content)}")

    if r2.status_code != 200:
        print("FAILED at Step 3")
        continue

    with open(info['filename'], 'wb') as fh:
        fh.write(r2.content)

    print(f"SUCCESS -> {info['filename']} ({len(r2.content):,} bytes)")


--- health_care ---
Step 1 status: 200  body: {"template_url":"https://goadmin-stage.ifrc.org/static/files/local_units/Health-Care-Bulk-Import-Template-Local-Units.xlsm"}
Step 2 template_url: https://goadmin-stage.ifrc.org/static/files/local_units/Health-Care-Bulk-Import-Template-Local-Units.xlsm
Step 3 status: 200  bytes received: 143871
SUCCESS -> output/template_health_care.xlsm (143,871 bytes)

--- local_unit ---
Step 1 status: 200  body: {"template_url":"https://goadmin-stage.ifrc.org/static/files/local_units/Administrative-Bulk-Import-Template-Local-Units.xlsx"}
Step 2 template_url: https://goadmin-stage.ifrc.org/static/files/local_units/Administrative-Bulk-Import-Template-Local-Units.xlsx
Step 3 status: 200  bytes received: 76348
SUCCESS -> output/template_local_unit.xlsm (76,348 bytes)


## 2. Inspect Template Structure
- **Row 1** → Category group headings
- **Row 2** → Column names (the actual field names we use for data)
- **Row 3** → Descriptions / mandatory flags
- **Sheet 'Options'** → Allowed values for dropdown columns

In [57]:
template_meta = {}  # Store {type: {col_name: {desc, options}}}

for info in templates_to_download:
    fp = info['filename']
    if not os.path.exists(fp):
        print(f"File not found: {fp}")
        continue

    print(f"\n{'='*60}")
    print(f" TEMPLATE: {info['type']}  ({fp})")
    print(f"{'='*60}")

    try:
        wb = openpyxl.load_workbook(fp, keep_vba=True, data_only=True)
    except Exception as e:
        print(f"Could not open file: {e}")
        continue

    # ---- Main data sheet ----
    ws = wb.active
    row1 = [c.value for c in ws[1]]   # Category groups
    row2 = [c.value for c in ws[2]]   # Column names  <-- used for data mapping
    row3 = [c.value for c in ws[3]]   # Descriptions / mandatory

    meta = {}
    print(f"\n{'Col#':<5} {'Category':<30} {'Field Name':<35} {'Description/Mandatory'}")
    print("-" * 110)
    for i, (cat, field, desc) in enumerate(zip(row1, row2, row3), 1):
        if field is not None:
            print(f"{i:<5} {str(cat or ''):<30} {str(field):<35} {str(desc or '')}")
            meta[str(field)] = {"col_index": i, "category": str(cat or ""), "description": str(desc or ""), "options": []}

    # ---- Options sheet ----
    if 'Options' in wb.sheetnames:
        ws_opt = wb['Options']
        print(f"\n--- Options Sheet ---")
        opt_headers = [c.value for c in ws_opt[1]]
        print(f"Option columns: {opt_headers}")
        for col_idx, col_name in enumerate(opt_headers, 1):
            if col_name is None:
                continue
            values = []
            for row in ws_opt.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    if cell.value is not None:
                        values.append(cell.value)
            print(f"  {col_name}: {values[:20]}")
            if str(col_name) in meta:
                meta[str(col_name)]['options'] = values

    template_meta[info['type']] = meta

print("\nInspection complete. Stored field metadata in 'template_meta'.")


 TEMPLATE: health_care  (output/template_health_care.xlsm)

Col#  Category                       Field Name                          Description/Mandatory
--------------------------------------------------------------------------------------------------------------
1     Administrative Details         Date of Update                      Required. DD/MM/YYYY
2                                    Local Unit Name (En)                Enter the Local Unit Name in English
3                                    Local Unit Name (Local)             Required. Enter the Local Unit Name in the local language
4                                    Visibility                          Required
5                                    Sub-type                            Example: Hospital
6                                    Affiliation                         Required. Select the Affiliation from the dropdown
7                                    Other Affiliation                   If 'Other' is selected in Af

## 3. Fill Templates with Sample Data for Germany

Data is appended from **row 4** onwards. Field names are taken from **row 2** of the template.

> After running cell 2 above, you can see all available field names and options printed. The data dictionaries below use those exact field names.

In [58]:
TODAY = datetime.today().strftime('%d/%m/%Y')
# 1. Load the Health Care records from the JSON file
health_care_records = []
hc_json_path = "output/health_care_upload.json"
if os.path.exists(hc_json_path):
    with open(hc_json_path, 'r', encoding='utf-8') as f:
        # Load the JSON and automatically inject TODAY's date if it says "TODAY"
        raw_data = f.read().replace('"TODAY"', f'"{TODAY}"')
        health_care_records = json.loads(raw_data)
        print(f"Loaded {len(health_care_records)} Health Care records.")
else:
    print(f"File not found: {hc_json_path}. Skipping.")


Loaded 5 Health Care records.


In [59]:
# 2. Load the Local Unit / Humanitarian records from the JSON file
local_unit_records = []
lu_json_path = "output/local_unit_upload.json"
if os.path.exists(lu_json_path):
    with open(lu_json_path, 'r', encoding='utf-8') as f:
        raw_data = f.read().replace('"TODAY"', f'"{TODAY}"')
        local_unit_records = json.loads(raw_data)
        print(f"Loaded {len(local_unit_records)} Local Unit records.")
else:
    print(f"File not found: {lu_json_path}. Skipping.")


Loaded 5 Local Unit records.


In [60]:
# Helper to find the true first empty row in Excel
def find_first_empty_row(ws, start_row=4):
    for row in range(start_row, ws.max_row + 2):
        # Check if every column in this row is None
        is_empty = all(ws.cell(row=row, column=c).value is None for c in range(1, ws.max_column + 1))
        if is_empty:
            return row
    return start_row


In [61]:
# 3. Define the filling function
def fill_and_save(input_file, output_file, records, header_row=2, data_start_row=4):
    if not os.path.exists(input_file):
        print(f"MISSING: {input_file}")
        return
    wb = openpyxl.load_workbook(input_file, keep_vba=True)
    ws = wb.active
    headers = [c.value for c in ws[header_row]]
    print(f"\nFilling {output_file}")
    
    # Calculate exactly where to start writing
    next_row = find_first_empty_row(ws, data_start_row)
    print(f"  Starting to write data perfectly at row {next_row}")
    for rec in records:
        row_data = []
        for header in headers:
            val = ""
            if header is not None:
                h_str = str(header).strip()
                h_key = h_str.lower().replace(" ", "_")
                # Dictionary matching
                if h_str in rec:
                    val = rec[h_str]
                elif h_key in rec:
                    val = rec[h_key]
                else:
                    for k, v in rec.items():
                        if k.lower().replace("_", " ") == h_str.lower():
                            val = v
                            break
            row_data.append(val)
        # Write the row data to the worksheet
        for col_idx, val in enumerate(row_data, 1):
            ws.cell(row=next_row, column=col_idx, value=val)
            
        print(f"  Written data row at Excel row {next_row}")
        next_row += 1
    wb.save(output_file)
    print(f"  Saved -> {output_file}")


In [62]:
# 4. Execute the fills!
if health_care_records:
    fill_and_save("output/template_health_care.xlsm", "output/filled_health_care.xlsm", health_care_records)
    
if local_unit_records:
    fill_and_save("output/template_local_unit.xlsm", "output/filled_local_unit.xlsm", local_unit_records)


Filling output/filled_health_care.xlsm
  Starting to write data perfectly at row 4
  Written data row at Excel row 4
  Written data row at Excel row 5
  Written data row at Excel row 6
  Written data row at Excel row 7
  Written data row at Excel row 8
  Saved -> output/filled_health_care.xlsm

Filling output/filled_local_unit.xlsm
  Starting to write data perfectly at row 4
  Written data row at Excel row 4
  Written data row at Excel row 5
  Written data row at Excel row 6
  Written data row at Excel row 7
  Written data row at Excel row 8
  Saved -> output/filled_local_unit.xlsm


## 4. Bulk Upload Filled Files to Staging

In [63]:
UPLOAD_URL =  f"{BASE_URL}/bulk-upload-local-unit/"
# Use the exact MIME type required for .xlsm files
MIME_TYPE = 'application/vnd.ms-excel.sheet.macroEnabled.12'
# We map the file to the corresponding local_unit_type ID (2 for Health Care, 4 for Humanities/Local Unit)
uploads = [
    {"file": "output/filled_health_care.xlsm", "type_id": 2},
    {"file": "output/filled_local_unit.xlsm", "type_id": 4}
]


In [64]:
for item in uploads:
    file_path = item["file"]
    type_id = item["type_id"]
    
    if os.path.exists(file_path):
        print(f"\nUploading: {file_path}")
        print(f"Targeting Country: 72 (Germany), Local Unit Type: {type_id}")
        
        with open(file_path, 'rb') as f:
            # 1. Attach the file
            files = {
                'file': (os.path.basename(file_path), f, MIME_TYPE)
            }
            
            # 2. Attach the required body parameters
            data = {
                'country': 72,                  # 72 is Germany
                'local_unit_type': type_id      # 2 or 4
            }
            
            # Send the request with BOTH files and data
            response = requests.post(UPLOAD_URL, headers=HEADERS, files=files, data=data)
            
            print(f"Status: {response.status_code}")
            if response.status_code == 201 or response.status_code == 200:
                print("SUCCESS")
                print(response.json())
            else:
                print("FAILED")
                try:
                    print(response.json())
                except:
                    print(response.text)
    else:
        print(f"\nSkipping {file_path} - file not found.")


Uploading: output/filled_health_care.xlsm
Targeting Country: 72 (Germany), Local Unit Type: 2
Status: 201
SUCCESS
{'id': 129, 'country_details': {'name': 'Germany', 'iso3': 'DEU', 'id': 72}, 'local_unit_type_details': {'name': 'Health Care', 'code': 2, 'id': 2, 'colour': '#011e41', 'image_url': 'https://goadmin-stage.ifrc.org/static/images/local_units/local_unit_type/Healthcare.png'}, 'triggered_by_details': {'id': 9634, 'username': 'arun.gandhi@ifrc.org', 'email': 'arun.gandhi@ifrc.org', 'first_name': 'Arun', 'last_name': 'Gandhi'}, 'file_name': 'filled_health_care.xlsm', 'error_message': None, 'status_details': 'Pending', 'success_count': 0, 'failed_count': 0, 'file_size': 146692, 'status': 3, 'triggered_at': '2026-03-04T18:47:52.582008Z', 'file': 'https://dsgofilestorage.blob.core.windows.net/api/local_unit/bulk_uploads/8eada51e3d5f426883f4c21ce5e8b39b/filled_health_care.xlsm', 'error_file': None, 'triggered_by': 9634}

Uploading: output/filled_local_unit.xlsm
Targeting Country: 72